### 1. Installation des dépendances
Nous avons besoin de `langchain` pour l'orchestration, `faiss-cpu` pour la recherche vectorielle, et `sentence-transformers` pour transformer le texte en vecteurs numériques.

In [1]:
!pip install -q -U langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers

### 2. Chargement et Découpage des documents
Pour cet exemple, nous allons créer des documents fictifs sur l'intelligence artificielle. Nous utilisons un `RecursiveCharacterTextSplitter` pour diviser le texte en morceaux (chunks) gérables.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Exemple de base de connaissances
knowledge_base = [
    "Le RAG (Retrieval-Augmented Generation) est une technique qui combine la récupération de documents et la génération de texte.",
    "FAISS est une bibliothèque développée par Facebook AI Research pour la recherche de similarité efficace dans des ensembles de vecteurs denses.",
    "Les transformeurs sont des modèles de deep learning qui utilisent des mécanismes d'attention pour traiter des données séquentielles.",
    "Hugging Face propose une vaste collection de modèles pré-entraînés pour le traitement du langage naturel (NLP)."
]

documents = [Document(page_content=text) for text in knowledge_base]

# Découpage du texte en segments plus petits
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

print(f'Nombre de segments créés : {len(texts)}')

Nombre de segments créés : 4


### 3. Création des Embeddings et de la base FAISS
Nous utilisons le modèle léger `all-MiniLM-L6-v2` pour convertir nos textes en vecteurs, puis nous les stockons dans un index FAISS.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Chargement du modèle d'embeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Création de la base de données vectorielle
vector_db = FAISS.from_documents(texts, embeddings)

print('Base de données vectorielle FAISS prête !')

/tmp/ipykernel_3195/1971501592.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base de données vectorielle FAISS prête !


### 4. Mise en place de la chaîne RAG
Nous allons configurer un pipeline simple qui :
1. Prend une question.
2. Cherche les documents les plus pertinents dans FAISS.
3. Utilise un modèle de Hugging Face pour générer une réponse basée sur ces documents.

In [4]:
import sys
# On force l'actualisation des chemins si nécessaire
if 'langchain' in sys.modules:
    import importlib
    importlib.reload(sys.modules['langchain'])

try:
    from langchain.chains import RetrievalQA
    from langchain_huggingface import HuggingFacePipeline
    from transformers import pipeline

    # Configuration du pipeline de génération
    pipe = pipeline("text-generation", model="gpt2", max_new_tokens=50, pad_token_id=50256)
    llm = HuggingFacePipeline(pipeline=pipe)

    # Création de la chaîne RAG
    rag_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_db.as_retriever(search_kwargs={"k": 2})
    )

    # Test final
    query = "Qu'est-ce que FAISS ?"
    response = rag_chain.invoke(query)

    print(f"Question: {query}")
    print(f"Réponse: {response['result']}")
except ModuleNotFoundError:
    print("Erreur : Le module 'langchain.chains' est introuvable.")
    print("ACTION REQUISE : Allez dans 'Exécution' -> 'Redémarrer la session', puis ré-exécutez les cellules 2, 3 et 4.")

Erreur : Le module 'langchain.chains' est introuvable.
ACTION REQUISE : Allez dans 'Exécution' -> 'Redémarrer la session', puis ré-exécutez les cellules 2, 3 et 4.
